# Databricks Vector Search — wikivoyage_index

This notebook demonstrates how to query the **wikivoyage_index** Delta Sync Vector Search index  
on Databricks using a hybrid similarity search with reranking.

**Index details**
| Property | Value |
|---|---|
| Index name | `workspace.default.wikivoyage_index` |
| Source table | `workspace.default.wikivoyage_search` |
| Serving endpoint | `wikivoyage_seach_endpoint` |
| Embedding model | `databricks-qwen3-embedding-0-6b` |
| Rows indexed | 21,363 |

## 1. Create Virtual Python Environment

Create and activate an isolated `.venv` virtual environment so all dependencies  
are installed without polluting the system Python.

In [ ]:
import sys, os

# Create a virtual environment in the project directory (skip if it already exists)
venv_path = os.path.join(os.getcwd(), ".venv")

if not os.path.isdir(venv_path):
    !python3 -m venv .venv
    print(f"Virtual environment created at: {venv_path}")
else:
    print(f"Virtual environment already exists at: {venv_path}")

print(f"Current Python executable: {sys.executable}")

## 2. Install Required Libraries

Install `databricks-vectorsearch` and `python-dotenv` into the virtual environment.

In [ ]:
# Install packages into the active kernel's environment
!pip install --quiet databricks-vectorsearch python-dotenv

print("Installation complete.")

## 3. Configure Environment Variables

Set your Databricks credentials. Values are read from a `.env` file if present,  
otherwise you can set them directly below.

> ⚠️ **Never commit real secrets to source control.** Use the `.env` file (already in `.gitignore`).

In [ ]:
import os
from dotenv import load_dotenv

# Load variables from .env file (if it exists)
load_dotenv(dotenv_path=os.path.join(os.getcwd(), ".env"), override=False)

# ── Override here only if NOT already set in the environment or .env ──
# os.environ.setdefault("WORKSPACE_URL",     "https://<your-workspace>.cloud.databricks.com")
# os.environ.setdefault("SP_CLIENT_ID",      "<your-service-principal-client-id>")
# os.environ.setdefault("SP_CLIENT_SECRET",  "<your-service-principal-client-secret>")

workspace_url     = os.environ.get("WORKSPACE_URL")
sp_client_id      = os.environ.get("SP_CLIENT_ID")
sp_client_secret  = os.environ.get("SP_CLIENT_SECRET")

# Validate that all required variables are present
missing = [k for k, v in {
    "WORKSPACE_URL":    workspace_url,
    "SP_CLIENT_ID":     sp_client_id,
    "SP_CLIENT_SECRET": sp_client_secret,
}.items() if not v]

if missing:
    raise EnvironmentError(
        f"Missing required environment variable(s): {', '.join(missing)}\n"
        "Edit the .env file in the project root and set the values there."
    )

print(f"WORKSPACE_URL    : {workspace_url}")
print(f"SP_CLIENT_ID     : {sp_client_id}")
print(f"SP_CLIENT_SECRET : {'*' * len(sp_client_secret)}")

## 4. Initialize Vector Search Client

Instantiate `VectorSearchClient` using the workspace URL and Service Principal credentials.

In [ ]:
from databricks.vector_search.client import VectorSearchClient
from databricks.vector_search.reranker import DatabricksReranker

vsc = VectorSearchClient(
    workspace_url=workspace_url,
    service_principal_client_id=sp_client_id,
    service_principal_client_secret=sp_client_secret,
)

print("VectorSearchClient initialized successfully.")

## 5. Connect to Vector Search Index

Retrieve the `workspace.default.wikivoyage_index` index object from the  
`wikivoyage_seach_endpoint` serving endpoint.

In [ ]:
ENDPOINT_NAME = "wikivoyage_seach_endpoint"
INDEX_NAME    = "workspace.default.wikivoyage_index"

index = vsc.get_index(
    endpoint_name=ENDPOINT_NAME,
    index_name=INDEX_NAME,
)

print(f"Connected to index : {INDEX_NAME}")
print(f"Endpoint           : {ENDPOINT_NAME}")

## 6. Perform Hybrid Similarity Search with Reranking

Run a **HYBRID** search (dense vector + BM25 keyword) and apply `DatabricksReranker`  
to re-score results based on the `content` column.

Edit `QUERY_TEXT` to test different search queries.

In [ ]:
import json

# ── Search parameters ──────────────────────────────────────────────────────────
QUERY_TEXT  = "best places to visit in Japan"   # ← change me
NUM_RESULTS = 3
COLUMNS     = ["content", "title"]

# ── Execute hybrid search with reranking ───────────────────────────────────────
results = index.similarity_search(
    num_results=NUM_RESULTS,
    columns=COLUMNS,
    query_text=QUERY_TEXT,
    query_type="HYBRID",
    reranker=DatabricksReranker(columns_to_rerank=["content"]),
)

# ── Display results ────────────────────────────────────────────────────────────
print(f"Query : '{QUERY_TEXT}'")
print(f"Top {NUM_RESULTS} results\n" + "=" * 60)

hits = results.get("result", {}).get("data_array", [])
columns_returned = results.get("manifest", {}).get("columns", [])
col_names = [c.get("name") for c in columns_returned]

for rank, hit in enumerate(hits, start=1):
    row = dict(zip(col_names, hit))
    print(f"\n── Result {rank} ──────────────────────────────────────")
    print(f"  Title   : {row.get('title', 'N/A')}")
    content_preview = str(row.get('content', ''))[:300]
    print(f"  Content : {content_preview}{'...' if len(str(row.get('content', ''))) > 300 else ''}")

# Raw response (uncomment to inspect)
# print(json.dumps(results, indent=2, default=str))